# Conformal 3D (G(4,1)) — Spheres, Circles, Point Pairs

**Part I · Geometric Algebra & Core** — Tutorial 06

This tutorial is a deep dive into `BasisN3`, the **conformal model** of 3D
Euclidean space. It appends two extra basis vectors — combined into the null
vectors `einf` (e∞) and `eo` (e₀) — so that Euclidean points, spheres, circles,
lines, and planes all become simple blades.

By the end you will be able to:

- Understand the **null-vector embedding** — why `einf` and `eo` square to zero.
- Distinguish **IPNS vs OPNS** representations and switch them with `N3.opns`.
- Build a **sphere** from a center and radius, and read center/radius back.
- Build **circles, point pairs, lines, and planes** as blades.
- Compute **intersections** (sphere ∩ sphere = circle, plane ∩ plane = line, …)
  with the `meet()` blade operation.
- Apply **rotors, translators, dilators, and inversions** with the sandwich product.

> **Prerequisites:** [Tutorial 03](../03_basis_classes/) (named blades) and
> [Tutorial 04](../04_euclidean_e3/) (rotors). Geometric entities and operators are
> created through the `pytanga.geometry` submodule.


## 1. Setup

The imports mirror the earlier deep dives, but with `BasisN3` and the conformal
entity types (`Sphere`, `Circle`, `PointPair`, …).

In [17]:
import math

from pytanga import MV
from pytanga.basis import BasisN3
from pytanga.geometry import (
    Circle,
    Dilator,
    Direction,
    Geometry,
    Inversion,
    Line,
    Plane,
    Point,
    PointPair,
    Rotor,
    Space,
    Sphere,
    Translator,
)

N3 = BasisN3()        # conformal 3D: G(5, 0b10000) — the 5D null embedding of G(4,1)
geo = Geometry(N3)    # binds the algebra; OPNS/IPNS read from N3.opns (default True)


## 2. The null-vector embedding

`BasisN3` adds two basis vectors to the three Euclidean ones: `ep` (e₄,
`ep² = +1`) and `em` (e₅, `em² = −1`). They are combined into the two **null
vectors** of the conformal model:

- `einf = ep + em` — the point at infinity (e∞)
- `eo = ½·em − ½·ep` — the origin (e₀)

Both square to zero, and their inner product is `−1`. The algebra is the
5-dimensional null-vector embedding of the conformal model **G(4,1)**.

In [7]:
ep, em = N3.ep, N3.em
einf, eo = N3.einf, N3.eo

ep.show("ep   (ep² = +1)")
em.show("em   (em² = −1)")
einf.show("einf = ep + em  (point at infinity)")
eo.show("eo   = ½·em − ½·ep  (origin)")

print("einf * einf =", einf * einf)
print("eo   * eo   =", eo * eo)
N3.ip(eo, einf).show("eo · einf  (= −1 by definition)")


ep   (ep² = +1): 0.5 einf - eo

em   (em² = −1): 0.5 einf + eo

einf = ep + em  (point at infinity): einf

eo   = ½·em − ½·ep  (origin): eo

einf * einf = 0
eo   * eo   = 0


eo · einf  (= −1 by definition): - 1

A conformal point is the null vector
`Cop(x) = x + ½·|x|²·einf + eo`. Here it is built by hand, then squared to
confirm it lies on the null cone.

In [8]:
def cop(x: float, y: float, z: float) -> MV:
    r2 = x * x + y * y + z * z
    return N3.multivector({"e1": x, "e2": y, "e3": z}) + einf * (0.5 * r2) + eo


P = cop(1, 2, 3)
P.show("Cop(1,2,3) = x + ½·|x|²·einf + eo")
print("P * P =", P * P, "  (a conformal point is null)")


Cop(1,2,3) = x + ½·|x|²·einf + eo: e1 + 2 e2 + 3 e3 + 7 einf + eo

P * P = 0   (a conformal point is null)


## 3. IPNS vs OPNS

Every conformal entity has two dual representations:

- **OPNS** (outer-product null space) — the object *is* the blade: a point is
  grade 1, a point pair grade 2, a circle grade 3, a sphere grade 4.
- **IPNS** (inner-product null space) — the dual form: a sphere or plane is
  grade 1, a circle or line grade 2, a point grade 4.

Which form `create()` builds is an **algebra property**, `N3.opns` (mutable,
default `True`). `analyze()` reads the same flag, so round-trips always agree.

In [9]:
p = geo.create(Point(2, 0, 0))
print("OPNS point grade:", p.grades)

N3.opns = False
p_ipns = geo.create(Point(2, 0, 0))
print("IPNS point grade:", p_ipns.grades)
N3.opns = True

s = geo.create(Sphere(center=Point(0, 0, 0), radius=2.0))
print("OPNS sphere grade:", s.grades)

N3.opns = False
s_ipns = geo.create(Sphere(center=Point(0, 0, 0), radius=2.0))
print("IPNS sphere grade:", s_ipns.grades)
N3.opns = True


OPNS point grade: [1]
IPNS point grade: [4]
OPNS sphere grade: [4]
IPNS sphere grade: [1]


## 4. Spheres — build and read back

A sphere is built from a `Point` center and a `float` radius. In OPNS it is a
grade-4 blade; in IPNS it is the grade-1 vector `S = c − ½·r²·einf`.

In [10]:
sphere = Sphere(center=Point(1, 2, 3), radius=5.0)
mv = geo.create(sphere)
mv.show("Sphere: OPNS grade-4 blade")

result = geo.analyze(mv)
print("center:", result.center)
print("radius:", result.radius)


Sphere: OPNS grade-4 blade: - 5.5 e123∧einf - e123∧eo - 3 e12∧einf∧eo + 2 e13∧einf∧eo - e23∧einf∧eo

center: Point(1.00, 2.00, 3.00)
radius: 5.0


In [11]:
N3.opns = False
mv_ipns = geo.create(sphere)
mv_ipns.show("The same sphere in IPNS: grade-1 vector c − ½·r²·einf")
print("analyze (IPNS):", geo.analyze(mv_ipns))
N3.opns = True


The same sphere in IPNS: grade-1 vector c − ½·r²·einf: e1 + 2 e2 + 3 e3 - 5.5 einf + eo

analyze (IPNS): Sphere(c=Point(1.00, 2.00, 3.00), r=5.00)


## 5. Circles, point pairs, lines, and planes

Each entity is a blade of a specific grade, built by wedging the right
conformal points:

| Entity | OPNS grade | Blade |
|---|---|---|
| Point | 1 | `Cop(p)` |
| Point pair | 2 | `Cop(a) ∧ Cop(b)` |
| Circle | 3 | `Cop(a) ∧ Cop(b) ∧ Cop(c)` |
| Line | 3 | `Cop(a) ∧ Cop(b) ∧ einf` |
| Plane | 4 | `Cop(a) ∧ Cop(b) ∧ Cop(c) ∧ einf` |
| Sphere | 4 | `Cop(a) ∧ Cop(b) ∧ Cop(c) ∧ Cop(d)` |

The `Geometry` class builds all of them from convenient constructors:

In [12]:
entities = [
    ("Point", Point(1, 2, 3)),
    ("Direction", Direction(0, 1, 0)),
    ("PointPair", PointPair(point_a=Point(-2, 0, 0), point_b=Point(2, 0, 0))),
    ("Line", Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0))),
    ("Circle", Circle(center=Point(0, 0, 0), normal=Direction(0, 0, 1), radius=2.0)),
    ("Plane", Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1))),
    ("Sphere", Sphere(center=Point(0, 0, 0), radius=2.0)),
    ("Space", Space()),
]

for name, e in entities:
    mv = geo.create(e)
    result = geo.analyze(mv)
    print(f"{name:10s} grade {mv.grades} -> {result} ({type(result).__name__})")


Point      grade [1] -> Point(1.00, 2.00, 3.00) (Point)
Direction  grade [1] -> Dir(0.00, 1.00, 0.00) (Direction)
PointPair  grade [2] -> PntPair(Point(-2.00, 0.00, 0.00), Point(2.00, 0.00, 0.00)) (PointPair)
Line       grade [3] -> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(1.00, 0.00, 0.00)) (Line)
Circle     grade [3] -> Circle(c=Point(0.00, 0.00, 0.00), r=2.00, n=Dir(-0.00, -0.00, 1.00)) (Circle)
Plane      grade [4] -> Plane(pt=Point(-0.00, -0.00, 3.00), n=Dir(-0.00, -0.00, 1.00)) (Plane)
Sphere     grade [4] -> Sphere(c=Point(-0.00, -0.00, -0.00), r=2.00) (Sphere)
Space      grade [5] -> Space(scale=1.0) (Space)


The same blades can be assembled explicitly with the outer product:

In [13]:
A = geo.create(Point(-2, 0, 0))
B = geo.create(Point(2, 0, 0))

pp = A ^ B
print("A ^ B                        ->", geo.analyze(pp), " grade", pp.grades)

line = A ^ B ^ N3.einf
print("A ^ B ^ einf                 ->", geo.analyze(line), " grade", line.grades)

c1 = geo.create(Point(2, 0, 0))
c2 = geo.create(Point(0, 2, 0))
c3 = geo.create(Point(-2, 0, 0))
circle = c1 ^ c2 ^ c3
print("c1 ^ c2 ^ c3                 ->", geo.analyze(circle), " grade", circle.grades)

plane = c1 ^ c2 ^ c3 ^ N3.einf
print("c1 ^ c2 ^ c3 ^ einf          ->", geo.analyze(plane), " grade", plane.grades)

s1 = geo.create(Point(2, 0, 0))
s2 = geo.create(Point(-2, 0, 0))
s3 = geo.create(Point(0, 2, 0))
s4 = geo.create(Point(0, 0, 2))
sphere4 = s1 ^ s2 ^ s3 ^ s4
print("s1 ^ s2 ^ s3 ^ s4           ->", geo.analyze(sphere4), " grade", sphere4.grades)


A ^ B                        -> PntPair(Point(-2.00, 0.00, 0.00), Point(2.00, 0.00, 0.00))  grade [2]
A ^ B ^ einf                 -> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(1.00, 0.00, 0.00))  grade [3]
c1 ^ c2 ^ c3                 -> Circle(c=Point(0.00, 0.00, 0.00), r=2.00, n=Dir(-0.00, -0.00, 1.00))  grade [3]
c1 ^ c2 ^ c3 ^ einf          -> Plane(pt=Point(-0.00, -0.00, 0.00), n=Dir(-0.00, -0.00, 1.00))  grade [4]
s1 ^ s2 ^ s3 ^ s4           -> Sphere(c=Point(0.00, 0.00, 0.00), r=2.00)  grade [4]


## 6. `join()` and `meet()` — union and intersection

For blades `A` and `B`:

- `join(A, B)` is the **union** — the smallest blade containing both.
- `meet(A, B)` is the **intersection** — the largest blade contained in both,
  computed as `meet(A, B) = dual(join(dual(A), dual(B)))`.

In the conformal model these read directly as geometric operations — e.g. the
meet of two spheres is their intersection circle.

In [14]:
P = geo.create(Point(1, 2, 3))
Q = geo.create(Point(-1, 0, 2))
S1 = geo.create(Sphere(center=Point(0, 0, 0), radius=2.0))
S2 = geo.create(Sphere(center=Point(1, 0, 0), radius=1.5))
pl0 = geo.create(Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)))
plY = geo.create(Plane(point=Point(0, 0, 0), normal=Direction(0, 1, 0)))

print("join(P, Q)   ->", geo.analyze(N3.join(P, Q)), " (two points span a point pair)")
print("meet(S1, S2) ->", geo.analyze(N3.meet(S1, S2)), " (sphere ∩ sphere = circle)")
print("meet(S1, pl0)->", geo.analyze(N3.meet(S1, pl0)), " (sphere ∩ plane = circle)")
print("meet(pl0, plY)->", geo.analyze(N3.meet(pl0, plY)), " (plane ∩ plane = line)")


join(P, Q)   -> PntPair(Point(1.00, 2.00, 3.00), Point(-1.00, -0.00, 2.00))  (two points span a point pair)
meet(S1, S2) -> Circle(c=Point(1.37, 0.00, 0.00), r=1.45, n=Dir(1.00, -0.00, -0.00))  (sphere ∩ sphere = circle)
meet(S1, pl0)-> Circle(c=Point(0.00, 0.00, 0.00), r=2.00, n=Dir(-0.00, -0.00, 1.00))  (sphere ∩ plane = circle)
meet(pl0, plY)-> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(-1.00, 0.00, 0.00))  (plane ∩ plane = line)


Contrast `join()`/`meet()` with the raw products:

- the **outer product** `∧` is the join of *independent* subspaces and gives
  **zero for incident** ones;
- the **inner product** `|` measures the metric and gives an **incidence test**
  (a point lies on an IPNS sphere exactly when their inner product is zero).

In [15]:
# Outer product: join for independent blades, zero for incident blades
print("P ^ Q is zero:", (P ^ Q).is_zero, "  (independent -> point pair)")
print("P ^ P is zero:", (P ^ P).is_zero, "  (incident -> zero)")

# Inner product: incidence of a point with an IPNS sphere
N3.opns = False
S_ipns = geo.create(Sphere(center=Point(0, 0, 0), radius=2.0))
N3.opns = True

P_on = geo.create(Point(2, 0, 0))
P_off = geo.create(Point(3, 0, 0))
print("ip(P_on,  S_ipns) =", N3.ip(P_on, S_ipns), "  (0 -> point on sphere)")
print("ip(P_off, S_ipns) =", N3.ip(P_off, S_ipns), "  (non-zero -> outside)")


P ^ Q is zero: False   (independent -> point pair)
P ^ P is zero: True   (incident -> zero)
ip(P_on,  S_ipns) = 0   (0 -> point on sphere)
ip(P_off, S_ipns) = -2.5   (non-zero -> outside)


## 7. N3 operators — rotors, translators, dilators, inversion

The conformal model represents all of these as **versors** (products of
invertible vectors). Apply one to a point with the versor (sandwich) product
`R·x·~R`, available as `N3.vp(R, x)`.

In [16]:
rot = geo.create(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
trans = geo.create(Translator(vector=Direction(2, 0, 0)))
dil = geo.create(Dilator(factor=2.0))
inv = geo.create(Inversion(center=Point(0, 0, 0)))

p = geo.create(Point(2, 0, 0))

print("rotate 90° about z  :", geo.analyze(N3.vp(rot, p)))
print("translate by (2,0,0) :", geo.analyze(N3.vp(trans, p)))
print("dilate by 2         :", geo.analyze(N3.vp(dil, p)))
print("invert (unit sphere):", geo.analyze(N3.vp(inv, p)))


rotate 90° about z  : Point(0.00, 2.00, 0.00)
translate by (2,0,0) : Point(4.00, 0.00, 0.00)
dilate by 2         : Point(4.00, 0.00, 0.00)
invert (unit sphere): Point(0.50, -0.00, -0.00)


## 8. Visual examples

Three equal-radius spheres intersect pairwise in a circle; all three together
intersect in a **point pair** (the two points where the circle meets the third
sphere). Viewer setup is covered in [Part II — Visualization](../../visualization/).

In [18]:
from pytanga.viz import LabelStyle, CircleStyle, PointPairStyle, SphereStyle, Visualizer

S1 = geo.create(Sphere(center=Point(-1, 0, 0), radius=2.0))
S2 = geo.create(Sphere(center=Point(1, 0, 0), radius=2.0))
S3 = geo.create(Sphere(center=Point(0, 0, 2), radius=2.0))

circle = N3.meet(S1, S2)           # sphere ∩ sphere = circle
point_pair = N3.meet(circle, S3)   # circle ∩ sphere = point pair

viz = Visualizer(title="N3 — three spheres, intersection circle and point pair")
for mv, color, label, opacity in [(S1, "#4488ff", "S1", 1.0), (S2, "#44cc44", "S2", 1.0), (S3, "#cc6666", "S3", 0.5)]:
    viz.new(mv, color=color, opacity=opacity,
             style=SphereStyle(wireframe=False), 
            tex_label=label)
viz.new(circle, color="#ffcc00", style=CircleStyle(tube_radius=0.2), 
        label="S1 ∩ S2 (circle)",
        label_style=LabelStyle(align=(0,0)))
viz.new(point_pair, color="#ff44ff", 
        style=PointPairStyle(point_size=0.5), 
        label="S1 ∩ S2 ∩ S3 (point pair)",
        label_style=LabelStyle(align=(1,0), along=1))

viz.display_snapshot()


A translator displaces a point; a dilator scales it about the origin:

In [ ]:
p = Point(1, 0, 0)
translated = geo.analyze(N3.vp(geo.create(Translator(vector=Direction(2, 0, 0))), geo.create(p)))
scaled = geo.analyze(N3.vp(geo.create(Dilator(factor=2.0)), geo.create(p)))

viz2 = Visualizer(title="N3 — translator and dilator applied to a point")
viz2.new(geo.create(p), color="#ffffff", label="p")
viz2.new(geo.create(translated), color="#44ff44", label="translated p")
viz2.new(geo.create(scaled), color="#ff44ff", label="dilated p")

viz2.display_snapshot()


Export both scenes as self-contained HTML:

In [14]:
import os

os.makedirs("_output/06_conformal_n3", exist_ok=True)
viz.export_snapshot("_output/06_conformal_n3/conformal_entities.html", overwrite=True)
viz2.export_snapshot("_output/06_conformal_n3/operators.html", overwrite=True)
print("Exported _output/06_conformal_n3/conformal_entities.html")
print("Exported _output/06_conformal_n3/operators.html")


Exported _output/06_conformal_n3/conformal_entities.html

Exported _output/06_conformal_n3/operators.html


## 9. Summary & next steps

You now know the conformal model `BasisN3`:

| Concept | API |
|---|---|
| Conformal point | `geo(Point(x, y, z))` |
| Sphere (build / read) | `Sphere(center, radius)` → `geo.analyze(...)` |
| Point pair / circle / line / plane | `PointPair(...)`, `Circle(...)`, `Line(...)`, `Plane(...)` |
| Union of subspaces | `N3.join(A, B)` |
| Intersection | `N3.meet(A, B)` |
| Incidence | `N3.ip(P, S_ipns)` / `P ^ S_opns` |
| Operators | `geo(Rotor/Translator/Dilator/Inversion(...))` → `N3.vp(R, x)` |

**Where to go next:**

- [**07 · PGA 3D**](../07_pga3/) — the plane-based sibling that shares the null
  embedding but with a single null vector `e0`.
- [**08 · Duality & Complements**](../08_duality/) — the `dual()`/`ldual()`/
  `complement()` family behind `meet()`.
